## GPT-2-medium Chess Fine-tuning

This notebook fine-tunes GPT-2-medium on Lichess chess games using LoRA to create a chess move prediction model. The fine-tuned model is pushed to HuggingFace as `chiaraDG/gpt2-medium-chess-lora`.

### Required Libraries



In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes python-chess huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_token_assignment_chess'))

### Imports

In [ ]:
import torch
import chess
import chess.pgn
import io
from datasets import load_dataset, Dataset # HF datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

### Load Lichess Dataset

In [ ]:
# Load dataset (streaming avoids huge memory use)
dataset = load_dataset(
    "Lichess/standard-chess-games",
    split="train",
    streaming = True # because it is a huge dataset
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/26138 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26138 [00:00<?, ?it/s]

In [ ]:
first_example = next(iter(dataset))
print(first_example.keys())

dict_keys(['Event', 'Site', 'White', 'Black', 'Result', 'WhiteTitle', 'BlackTitle', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'UTCDate', 'UTCTime', 'ECO', 'Opening', 'Termination', 'TimeControl', 'movetext'])


### Filter strong players

To use only high-quality games

In [ ]:
def filter_high_elo(example): # filtering function
    try:
        return int(example["WhiteElo"]) > 2000 and int(example["BlackElo"]) > 2000
    except:
        return False

In [ ]:
dataset = dataset.filter(filter_high_elo)

Select subset of dataset

In [ ]:
dataset = dataset.shuffle(seed=42, buffer_size = 10000) # shuffle dataset, buffer is small by default so we increase it here
dataset = dataset.take(20000) # could be increased in training works

### Convert PGN to FEN, UCI move

The loaded dataset stores the moves in PGN (Portable Game Notation) format, so it first needs to be reformatted to UCI moves.

In [ ]:
def game_to_examples(example):
    all_texts = [] # will hold expanded prompt strings

    # Loop over every PGN string in batch
    for pgn_string in example["movetext"]:

        # Wrap string in io.StringIO so parser can read it
        pgn_io = io.StringIO(pgn_string)

        # Parse game
        game = chess.pgn.read_game(pgn_io)

        # Safety check (skip if corrupted)
        if game is None:
            continue

        board = game.board()

        # Iterate through clean, extracted moves
        for move in game.mainline_moves():
            fen = board.fen()
            move_uci = move.uci()

            # Create exact training string
            training_text = f"Board FEN: {fen}\nBest move: {move_uci}"

            # Push move to board
            all_texts.append(training_text)

            board.push(move)
    return {"text": all_texts}

In [ ]:
processed_dataset = dataset.map(
    game_to_examples,
    remove_columns=['Event', 'Site', 'White', 'Black', 'Result', 'WhiteTitle',
                    'BlackTitle', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff',
                    'BlackRatingDiff', 'UTCDate', 'UTCTime', 'ECO', 'Opening',
                    'Termination', 'TimeControl', 'movetext'],
    batched=True
)

# processed_dataset = processed_dataset.flatten_indices()

### Modify tokenizer before fine-tuning

Adding chess-specific tokens

In [ ]:
model_name = "openai-community/gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add padding token
tokenizer.pad_token = tokenizer.eos_token # gpt-2 does not have padding token by default

# Add square tokens like a1, b1, etc.
square_tokens = [
    f"{file}{rank}"
    for file in "abcdefgh"
    for rank in "12345678"
]

tokenizer.add_tokens(square_tokens)

# Add special tokens for formatting
tokenizer.add_special_tokens({"additional_special_tokens": ["<FEN>", "<MOVE>"]})

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2

Now squares are atomic tokens, moves become more consistent and the models learns structure better.

### Load Model & Resize Embeddings

Because of adding the tokens, the embedding matrix must grow.

In [ ]:

model = AutoModelForCausalLM.from_pretrained(model_name)

# Resize model embeddings to match new tokenizer size
model.resize_token_embeddings(len(tokenizer))

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50323, 1024)

### Apply LoRA
Only trains small adapters, is fast and memory efficient.

In [ ]:

lora_config = LoraConfig(
    r=32, # rank
    lora_alpha=64, # 2x rank
    target_modules=["c_attn"], # GPT-2 attention
    lora_dropout=0.1,
    bias = "none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


### Tokenize Training Data

In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding = "max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [ ]:
tokenized_dataset = processed_dataset.map(tokenize_function, batched=True)

In [ ]:
# count = 0
# for example in tokenized_dataset:
#     count += 1
# print(f"Total training examples: {count}")

### Training Setup

In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2-medium-chess",
    per_device_train_batch_size=8,
    max_steps = 10000,
    logging_steps=100,
    save_steps=500,
    fp16 = True,
    learning_rate = 5e-4,
    warmup_steps = 100,
    weight_decay = 0.01
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.512166
200,2.368389
300,1.983796
400,2.011100
500,1.904043
600,1.802663
700,1.828549
800,1.854854
900,1.779391
1000,1.642738


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetunin

TrainOutput(global_step=10000, training_loss=1.5481347518920898, metrics={'train_runtime': 2525.4072, 'train_samples_per_second': 31.678, 'train_steps_per_second': 3.96, 'total_flos': 1.876728741888e+16, 'train_loss': 1.5481347518920898, 'epoch': 1.0})

## Push to Hugging Face

In [ ]:
model = model.merge_and_unload()

In [ ]:
# Check
from peft import PeftModel
print(type(model))
print(model.__class__.__name__)

<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
GPT2LMHeadModel


In [ ]:
tokenizer.save_pretrained("./gpt2-medium-chess")
model.save_pretrained("./gpt2-medium-chess")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Card for Hugging Face:


In [ ]:
model_card = """
---
language: en

tags:
  - chess
  - gpt2
  - lora
  - fine-tuned

---

# GPT-2 Chess LoRA

A GPT-2 model fine-tuned on Lichess chess games to predict the best move given a board position in FEN format.

## Training Data
- Dataset: Lichess/standard-chess-games
- Filtered to games where both players are rated > 2000 ELO
- 20000 games used for fine-tuning

## Prompt Format
The model expects input in exactly this format:
```
Board FEN: <fen string>
Best move:
```
And will complete it with a UCI move like `e2e4`.

## Training
- Base model: openai-community/gpt2-medium
- Fine-tuning method: LoRA (r=32, alpha=64)
- Max steps: 10000
- Learning rate: 5e-4
- Custom tokens added: all 64 board squares as atomic tokens (a1-h8)

## Usage
This model is used as part of a TransformerPlayer chess agent that scores all
legal moves by log-probability and selects the highest scoring one.
"""

with open("README.md", "w") as f:
    f.write(model_card)



In [ ]:
# Quick inference test before pushing
test_fen = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1"
prompt = f"Board FEN: {test_fen}\nBest move:"

device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=6)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Board FEN: rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1
Best move: ntilanwhile the cumbersccording


In [ ]:
# Test
test_fen = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1"
board = chess.Board(test_fen)
legal_moves = list(board.legal_moves)

prompt = f"Board FEN: {test_fen}\nBest move:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
prompt_len = inputs["input_ids"].shape[1]

scores = {}
for move in legal_moves:
    move_text = " " + move.uci()
    move_ids = tokenizer(move_text, return_tensors="pt").to(device)

    full_ids = torch.cat([inputs["input_ids"], move_ids["input_ids"]], dim=1)

    with torch.no_grad():
        outputs = model(input_ids=full_ids)
        logits = outputs.logits

    move_logits = logits[0, prompt_len - 1 : prompt_len - 1 + move_ids["input_ids"].shape[1], :]
    log_probs = torch.log_softmax(move_logits, dim=-1)
    token_log_probs = log_probs.gather(1, move_ids["input_ids"].T).squeeze()
    scores[move.uci()] = token_log_probs.sum().item()

# Print top 5 moves by model score
for move, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"{move}: {score:.4f}")

d7d6: -21.7129
d7d5: -21.7154
b7b6: -21.7225
h7h5: -21.7230
a7a5: -21.7237


### Push to HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

token=userdata.get('HF_token_assignment_chess')

model.push_to_hub("chiaraDG/gpt2-medium-chess-lora", token=token)
tokenizer.push_to_hub("chiaraDG/gpt2-medium-chess-lora", token=token)

from huggingface_hub import HfApi
api = HfApi(token=token)
api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id="chiaraDG/gpt2-medium-chess-lora",
    repo_type="model"
)

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...9oq8sy3/model.safetensors:   1%|          | 9.68MB / 1.42GB            

CommitInfo(commit_url='https://huggingface.co/chiaraDG/gpt2-medium-chess-lora/commit/ec943bd87f1758edc88afd757778d7cce49e6403', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='ec943bd87f1758edc88afd757778d7cce49e6403', pr_url=None, repo_url=RepoUrl('https://huggingface.co/chiaraDG/gpt2-medium-chess-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='chiaraDG/gpt2-medium-chess-lora'), pr_revision=None, pr_num=None)

###